In [ ]:
# !pip install dali-dataset

This notebook creates the raw audio dataset for tracks in the DALI dataset. DALI claims to have a built in function for this use, but it failed for me repeatedly.

In [78]:
import DALI as dali_code
import pandas as pd
import os
import re
from pydub import AudioSegment

In [ ]:
# Relative path to DALI folder
path = "../DALI_v1.0"

# Import DALI dataset
full_path = os.path.abspath(path)
dali_data = dali_code.get_the_DALI_dataset(full_path, skip=[], keep=[])

# Get info
dali_info_path = os.path.join(full_path, 'info/DALI_DATA_INFO.gz')
dali_info = dali_code.get_info(dali_info_path)

df_DALI = pd.DataFrame(dali_info[1:], columns=dali_info[0])

Function for downloading youtube audio. Requires ffmpeg, which requires an external download (for me at least). Be sure to keep track of where you've installed ffmpeg, ydl requires being told that location explicitly.

In [ ]:
import yt_dlp

def download_youtube_audio(url, filename, output_path='.', file_format='mp3'):
    """
    Downloads audio from a YouTube URL.

    Args:
        url (str): The YouTube video URL.
        output_path (str): The directory to save the audio file.
        file_format (str): The desired audio format (e.g., 'mp3', 'wav', 'm4a', 'opus').
    """
    # Create the output directory if it doesn't exist
    os.makedirs(output_path, exist_ok=True)

    ## IMPORTANT: put the name of your local ffmpeg directory
    if os.name == 'nt':
        ffmpeg_loc = "C:\\ffmpeg\\bin\\"
    
    ydl_opts = {
        'format': 'bestaudio/best',  # Download the best available audio format
        'outtmpl': os.path.join(output_path, f"{filename}"), # Output file template
        'postprocessors': [{
            'key': 'FFmpegExtractAudio', # Use FFmpeg to extract audio
            'preferredcodec': file_format, # Preferred audio codec
            'preferredquality': '0',     # Best quality
        }],
        'ffmpeg_location': ffmpeg_loc,
        'quiet': True, # Turns off text info
    }

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info_dict = ydl.extract_info(url, download=True)
            #title = info_dict.get('title', 'unknown_title')
            #print(f"Successfully downloaded: {filename}.{file_format}")
            return os.path.join(output_path, f"{filename}.{file_format}") # Return the downloaded file path
        
    except yt_dlp.DownloadError as e:
        print(f"Error downloading audio: {e}")
        return None
    
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None

Automatically download, if available, the youtube audio from each entry in the DALI database

In [ ]:
import pickle

def save_dictionary(data_dict, filepath):
    with open(filepath, 'wb') as f: # 'wb' mode for writing in binary
            pickle.dump(data_dict, f)
    return

def remove_punctuation(s):
    s = re.sub(r'[^a-zA-Z0-9\s]', '', s)
    return s.lower()

Be warned: This will take a long time to complete

In [ ]:
for entry_id in df_DALI['DALI_ID']:
    entry = dali_data[entry_id]
    output_dir = os.path.abspath(f"../DALI-audio/{entry.info['id']}")
    
    fname = remove_punctuation(entry.info['title']).replace(" ", "-") ### Check this

    ## Create directory
    os.makedirs(output_dir, exist_ok=True)

    ## Get audio
    youtube_url = entry.info['audio']['url']

    downloaded_file = download_youtube_audio(url=youtube_url, 
                                            filename=fname,
                                            output_path=output_dir,
                                            file_format='wav')

    # If the youtube download fails remove the directory and continue
    if not downloaded_file:
        os.rmdir(output_dir)
        continue

    save_dictionary(entry.info, os.path.abspath(f"../DALI-audio/{entry.info['id']}/metadata.pkl"))